In [37]:

from typing import TypedDict

from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import StateGraph, START, END

from langchain_community.tools import DuckDuckGoSearchRun
load_dotenv()

True

In [38]:




llm = ChatGroq(model="qwen/qwen3-32b", temperature=0.4, reasoning_format="hidden")

search_tool = DuckDuckGoSearchRun()

tools = [search_tool]

llm_with_tool = llm.bind_tools(tools)

In [39]:
class JokeState(TypedDict):
    topic: str
    joke: str
    explanation: str

In [40]:
def generate_joke(state: JokeState) -> JokeState:
    response = llm.invoke(
        f"Generate a funny and original joke on the topic: {state["topic"]}. Reply with only the joke. Dont exceed more than 1 line")
    return {"joke": response.content}

In [41]:
def explain_joke(state: JokeState) -> JokeState:
    response = llm.invoke(f"Explain the joke: {state["joke"]}. Reply with only joke explaination")
    return {"explanation": response.content}

In [42]:
graph = StateGraph(JokeState)

graph.add_node("generate_joke", generate_joke)
graph.add_node("explain_joke", explain_joke)

graph.add_edge(START, "generate_joke")
graph.add_edge("generate_joke", "explain_joke")
graph.add_edge("explain_joke", END)

In [43]:
checkpointer = InMemorySaver()
workflow = graph.compile(checkpointer=checkpointer)

In [44]:
initial_state = {"topic": "AI"}
thread_id = "thread_1"
config = {"configurable": {"thread_id": thread_id}}
final_state = workflow.invoke(input=initial_state, config=config)

print(final_state["joke"])
print("\n")
print(final_state["explanation"])


# print("\n\n")
# final_state = workflow.get_state(config=config).values
# print(final_state["joke"])

# final_state["joke"]


Why did the self-driving car act like a toddler at the grocery store? It kept stopping to "look at the snacks" and asked, "Are we algorithm-yet?"


The joke humorously compares a self-driving car's algorithmic behavior to a toddler's impatience. The car "stops to look at the snacks" as a playful metaphor for its sensors/programming pausing to process stimuli (like a toddler fixating on snacks). The pun "Are we algorithm-yet?" twists the classic "Are we there yet?" complaint, substituting "algorithm" to mock the car's reliance on its programming to navigate, mirroring a toddler's endless questioning. The humor lies in blending tech logic with human (toddler) behavior.


In [45]:
# list(workflow.get_state_history(config))

print("=== State History ===")
for snapshot in workflow.get_state_history(config):
    print(f"--- Step {snapshot.metadata['step']} | {snapshot.created_at} ---")
    print("Next:  ", snapshot.next)
    print("Values:", {k: v[:60] + "..." if isinstance(v, str) and len(v) > 60 else v
                      for k, v in snapshot.values.items()})
    print()


=== State History ===
--- Step 2 | 2026-06-21T15:57:24.941976+00:00 ---
Next:   ()
Values: {'topic': 'AI', 'joke': 'Why did the self-driving car act like a toddler at the groce...', 'explanation': "The joke humorously compares a self-driving car's algorithmi..."}

--- Step 1 | 2026-06-21T15:57:23.316812+00:00 ---
Next:   ('explain_joke',)
Values: {'topic': 'AI', 'joke': 'Why did the self-driving car act like a toddler at the groce...'}

--- Step 0 | 2026-06-21T15:57:18.758859+00:00 ---
Next:   ('generate_joke',)
Values: {'topic': 'AI'}

--- Step -1 | 2026-06-21T15:57:18.758023+00:00 ---
Next:   ('__start__',)
Values: {}



In [46]:
workflow.invoke(None, config=config)

{'topic': 'AI',
 'joke': 'Why did the self-driving car act like a toddler at the grocery store? It kept stopping to "look at the snacks" and asked, "Are we algorithm-yet?"',
 'explanation': 'The joke humorously compares a self-driving car\'s algorithmic behavior to a toddler\'s impatience. The car "stops to look at the snacks" as a playful metaphor for its sensors/programming pausing to process stimuli (like a toddler fixating on snacks). The pun "Are we algorithm-yet?" twists the classic "Are we there yet?" complaint, substituting "algorithm" to mock the car\'s reliance on its programming to navigate, mirroring a toddler\'s endless questioning. The humor lies in blending tech logic with human (toddler) behavior.'}